In [1]:
import pandas as pd
import numpy as np
import joblib
import os

In [2]:
# Load Data and Models
print("Loading Validation Data and Champion Models...")
model_data = pd.read_parquet("../data/processed/preprocessed_data_enriched.parquet")
model_data["time_step"] = (model_data["date"] - model_data["date"].min()).dt.days

# We only need the validation period (the future we are predicting)
max_date = model_data["date"].max()
split_date = max_date - pd.Timedelta(days=28)
val_data = model_data[model_data["date"] >= split_date].copy()

# Loading the saved models
lr_model = joblib.load("../models/lr_trend_model_xgb.pkl")
xgb_model = joblib.load("../models/xgboost_residual_model.pkl")

Loading Validation Data and Champion Models...


In [3]:
# Generating Predictions

print("Generating Demand Forecasts...")
drop_cols = ['id', 'd', 'date', 'sales', 'wm_yr_wk', 'time_step']
features = [col for col in val_data.columns if col not in drop_cols]

# Predicting the trend
lr_pred = lr_model.predict(val_data[['time_step']])

# Predicting the residuals
xgb_resid_pred = xgb_model.predict(val_data[features])

val_data["forecast_demand"] = lr_pred + xgb_resid_pred
val_data["forecast_demand"] = val_data['forecast_demand'].clip(lower=0)

Generating Demand Forecasts...


In [5]:
# Calculating Item-Specific RMSE
print("Calculating Item-Specific Error Rates...")

val_data['squared_error'] = (val_data['sales'] - val_data['forecast_demand'])**2

inventory_policy = val_data.groupby('item_id').agg(
    avg_daily_demand=('forecast_demand', 'mean'),
    item_rmse=('squared_error', lambda x: np.sqrt(x.mean()))
).reset_index()

Calculating Item-Specific Error Rates...


In [6]:
# Supply Chain Math (ROP & Safety Stock)
print("Calculating Safety Stock and Reorder Points...")

LEAD_TIME_DAYS = 7
Z_SCORE = 1.65 # 95% service level (Standard Walmart target)

# Safety Stock
inventory_policy['safety_stock'] = np.ceil(
    Z_SCORE * inventory_policy['item_rmse'] * np.sqrt(LEAD_TIME_DAYS)
)

# Reorder Point
inventory_policy['reorder_point'] = np.ceil(
    (inventory_policy['avg_daily_demand'] * LEAD_TIME_DAYS) + inventory_policy['safety_stock']
)

Calculating Safety Stock and Reorder Points...


In [7]:
# Final Export
item_encoder = joblib.load("../models/item_encoder.pkl")
inventory_policy['product_name'] = item_encoder.inverse_transform(inventory_policy['item_id'])

final_report = inventory_policy[[
    'product_name', 'avg_daily_demand', 'item_rmse', 'safety_stock', 'reorder_point'
]].sort_values(by='avg_daily_demand', ascending=False)

os.makedirs("../reports", exist_ok=True)
final_report.to_csv("../reports/daily_reorder_policy.csv", index=False)

final_report.to_csv("../reports/daily_reorder_policy.csv", index=False)

In [8]:
print(final_report.head(10))

     product_name  avg_daily_demand  item_rmse  safety_stock  reorder_point
702   FOODS_3_090         55.489812  17.555358          77.0          466.0
1198  FOODS_3_586         39.954493   8.425714          37.0          317.0
732   FOODS_3_120         39.081834  27.320679         120.0          394.0
864   FOODS_3_252         38.226719   9.153049          40.0          308.0
1199  FOODS_3_587         26.469252   7.192491          32.0          218.0
1113  FOODS_3_501         26.451947  13.913508          61.0          247.0
907   FOODS_3_295         25.362980  13.758675          61.0          239.0
676   FOODS_3_064         24.261922   8.450701          37.0          207.0
894   FOODS_3_282         23.831107  12.577305          55.0          222.0
1123  FOODS_3_511         22.245104   5.354388          24.0          180.0
